Filtragem e reconhecimento dos dados de filmes

In [1]:
import pandas as pd

df = pd.read_csv('dataset/movies.csv')

#Tratando os valores Null
df['homepage'] = df['homepage'].fillna(' ')
df['genres'] = df['genres'].fillna('Outro')
df['tagline'] = df['tagline'].fillna(' ')
df['cast'] = df['cast'].fillna(' ')
df['keywords'] = df['keywords'].fillna('Sem')
df['director'] = df['director'].fillna(' ')
df['release_date'] = df['release_date'].fillna('09/09/2009')
df['overview'] = df['overview'].fillna(' ')
df['runtime'] = df['runtime'].fillna('Sem')

#criando uma coluna nova "perfil"
df['perfil'] = df['genres'] + ' ' + df['overview'] + ' ' + df['keywords'] 
df['perfil'] = df['perfil'].str.lower()

Vetorizando os dados

In [2]:
#Transformando palavras em vetores para o entendimento da maquina
from sklearn.feature_extraction.text import TfidfVectorizer

#ajustes no vetorizador
vectorizador = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1,2)
)

#indicando qual coluna deve ser vetorizada
tfidf_matrix = vectorizador.fit_transform(df['perfil'])
#printando o array
vectorizador.get_feature_names_out()

array(['000', '007', '10', ..., 'zombies', 'zone', 'zoo'],
      shape=(5000,), dtype=object)

Procurando similaridades entre os filmes e transformando o array do vetor em lista

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

#procura similaridade entre os filmes
similaridade = cosine_similarity(tfidf_matrix)

#transformar a similaridade em lista, ordenar inversamente e ignorar o primeiro valor
listasimilaridade = list(enumerate(similaridade[0]))
listasimilaridade = sorted(listasimilaridade, key=lambda x: x[1], reverse=True)

#printar similaridade
for filme, valor in listasimilaridade[1:6]:
    print(f'{df["original_title"][filme]} = {valor:.2f}')

Moonraker = 0.27
Lifeforce = 0.25
Lost in Space = 0.24
Mad Max Beyond Thunderdome = 0.23
Silent Running = 0.21


In [4]:
#buscar o filme digitado e ver se tem ou não na lista

def recomendar(filme):
    filtro = df[df['original_title'].str.contains(filme, case=False, na=False)].head(1)
    if filtro.empty:
        return {"erro": 'Não encontrei.'}
    else:
        indice = filtro.index[0]
        sim = list(enumerate(similaridade[indice]))
        sim = sorted(sim, key=lambda x: x[1], reverse=True)
        resultados = []
        for idx, valor in sim[1:6]:
            resultados.append({
                'titulo': df['original_title'][idx],
                'score': round(valor, 2)
            })
        return resultados

recomendar('horror')

    


[{'titulo': 'A Night at the Roxbury', 'score': np.float64(0.24)},
 {'titulo': 'My Fair Lady', 'score': np.float64(0.21)},
 {'titulo': 'Out of the Dark', 'score': np.float64(0.21)},
 {'titulo': 'Anacondas: The Hunt for the Blood Orchid',
  'score': np.float64(0.18)},
 {'titulo': 'Bee Movie', 'score': np.float64(0.18)}]